<!--
 Licensed to the Apache Software Foundation (ASF) under one or more
 contributor license agreements.  See the NOTICE file distributed with
 this work for additional information regarding copyright ownership.
 The ASF licenses this file to You under the Apache License, Version 2.0
 (the "License"); you may not use this file except in compliance with
 the License.  You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing, software
 distributed under the License is distributed on an "AS IS" BASIS,
 WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 See the License for the specific language governing permissions and
 limitations under the License.
-->

In [1]:
import findspark
findspark.init()

from pyspark import SparkConf, SparkContext
nativesql_jars = "/path/to/gluten-XXXX.jar"
conf = SparkConf().setAppName("PySpark Gluten").setMaster("yarn")
conf.set("spark.executor.instances", "2")
conf.set("spark.executor.memory", "6g")
conf.set("spark.executor.cores", "2")
conf.set("spark.driver.memory", "2g")
conf.set("spark.memory.offHeap.enabled", "true")
conf.set("spark.memory.offHeap.size", "2g")
conf.set("spark.executor.memoryOverhead", "384M")
conf.set("spark.driver.extraClassPath", nativesql_jars)
conf.set("spark.executor.extraClassPath", nativesql_jars)
conf.set("spark.plugins", "org.apache.gluten.GlutenPlugin")
conf.set("spark.gluten.loadLibFromJar", "false")
conf.set("spark.shuffle.manager", "org.apache.spark.shuffle.sort.ColumnarShuffleManager")
sc = SparkContext(conf=conf)
from pyspark.sql import SparkSession
spark_session = SparkSession(sc)


In [2]:
df = spark_session.createDataFrame(
    [
        ("Normal", "Cellphone", 6000),
        ("Normal", "Tablet", 1500),
        ("Mini", "Tablet", 5500),
        ("Mini", "Cellphone", 5000),
        ("Foldable", "Cellphone", 6500),
        ("Foldable", "Tablet", 2500),
        ("Pro", "Cellphone", 3000),
        ("Pro", "Tablet", 4000),
        ("Pro Max", "Cellphone", 4500)
    ],
    ["product", "category", "revenue"]
)
df.filter("category = 'Cellphone'").explain()

== Physical Plan ==
VeloxColumnarToRowExec
+- ^(1) FilterExecTransformer (isnotnull(category#1) AND (category#1 = Cellphone))
   +- ^(1) InputIteratorTransformer[product#0, category#1, revenue#2L]
      +- ^(1) InputAdapter
         +- ^(1) RowToVeloxColumnar
            +- *(1) Scan ExistingRDD[product#0,category#1,revenue#2L]


